# SAM3 Segmentation Playground

Experiment with Meta's SAM3 on satellite imagery of properties.
SAM3 adds **text-prompted segmentation** on top of SAM2's point/box prompts.

**Prerequisites:** Accept the SAM3 license at [huggingface.co/facebook/sam3](https://huggingface.co/facebook/sam3) and login via `huggingface-cli login`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import requests
from io import BytesIO
import torch
from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

GOOGLE_MAPS_API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '')

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float32 if device == 'mps' else torch.bfloat16

print(f'API key loaded: {bool(GOOGLE_MAPS_API_KEY)}')
print(f'Device: {device}')
print(f'Dtype: {dtype}')

## 1. Fetch satellite image at different zoom levels

In [ ]:
def fetch_satellite(lat: float, lon: float, zoom: int = 19, size: str = '640x640') -> Image.Image:
    """Fetch a Google Maps satellite image."""
    url = (
        f'https://maps.googleapis.com/maps/api/staticmap'
        f'?center={lat},{lon}&zoom={zoom}&size={size}'
        f'&maptype=satellite&key={GOOGLE_MAPS_API_KEY}'
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert('RGB')


# ── CHANGE THESE ──
LAT, LON = 43.6262337, -79.74174239999999  # 1251 Quest Circle, Mississauga
ZOOM_LEVELS = [17, 18, 19, 20]

images = {}
fig, axes = plt.subplots(1, len(ZOOM_LEVELS), figsize=(5 * len(ZOOM_LEVELS), 5))
for ax, zoom in zip(axes, ZOOM_LEVELS):
    img = fetch_satellite(LAT, LON, zoom=zoom)
    images[zoom] = img
    ax.imshow(img)
    ax.set_title(f'Zoom {zoom}')
    ax.axis('off')
plt.suptitle(f'Satellite imagery at ({LAT:.4f}, {LON:.4f})', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Load SAM3 models

SAM3 has two image model types:
- **Sam3Model** — text-prompted concept segmentation (new in SAM3)
- **Sam3TrackerModel** — point/box-prompted segmentation (drop-in SAM2 replacement)

In [ ]:
from transformers import Sam3Model, Sam3Processor, Sam3TrackerModel, Sam3TrackerProcessor

# Text-prompted model (Promptable Concept Segmentation)
sam3_model = Sam3Model.from_pretrained('facebook/sam3').to(device)
sam3_processor = Sam3Processor.from_pretrained('facebook/sam3')

# Point/box-prompted model (Promptable Visual Segmentation — SAM2 drop-in)
sam3_tracker = Sam3TrackerModel.from_pretrained('facebook/sam3').to(device)
sam3_tracker_processor = Sam3TrackerProcessor.from_pretrained('facebook/sam3')

print(f'SAM3 models loaded on {device}')

## 3. Text-prompted segmentation (NEW in SAM3)

Describe what you want to segment using natural language — no points or boxes needed.
SAM3 understands 270K+ concepts.

In [ ]:
def segment_with_text(
    image: Image.Image,
    text: str,
    threshold: float = 0.5,
    mask_threshold: float = 0.5,
    title: str = '',
):
    """Segment an image using a text prompt."""
    inputs = sam3_processor(images=image, text=text, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold=threshold,
        mask_threshold=mask_threshold,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    masks = results['masks']
    scores = results['scores']
    boxes = results.get('boxes', [])

    n_masks = len(masks)
    n_cols = min(n_masks + 1, 5)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]

    # Original image
    axes[0].imshow(image)
    axes[0].set_title(f'Text: "{text}"')
    axes[0].axis('off')

    # Individual masks
    for i in range(min(n_masks, n_cols - 1)):
        mask = masks[i]
        mask_bool = mask.cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.array(mask).astype(bool)
        score_val = float(scores[i])

        axes[i + 1].imshow(image)
        overlay = np.zeros((*mask_bool.shape[:2], 4), dtype=np.float32)
        overlay[mask_bool] = [0.2, 0.6, 1.0, 0.5]
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f'Instance {i+1} (score: {score_val:.3f})')
        axes[i + 1].axis('off')

    display_title = title or f'Text segmentation: "{text}"'
    plt.suptitle(f'{display_title} — {n_masks} instance(s) found', fontsize=14)
    plt.tight_layout()
    plt.show()

    return results


# ── TRY DIFFERENT TEXT PROMPTS ──
img = images[19]

segment_with_text(img, 'rooftop')
segment_with_text(img, 'driveway')
segment_with_text(img, 'swimming pool')
segment_with_text(img, 'tree')

## 4. Compare text prompts across zoom levels

In [ ]:
# ── CHANGE THE TEXT PROMPT ──
TEXT_PROMPT = 'rooftop'

fig, axes = plt.subplots(2, len(ZOOM_LEVELS), figsize=(5 * len(ZOOM_LEVELS), 10))

for col, zoom in enumerate(ZOOM_LEVELS):
    img = images[zoom]
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Zoom {zoom}')
    axes[0, col].axis('off')

    inputs = sam3_processor(images=img, text=TEXT_PROMPT, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    axes[1, col].imshow(img)
    overlay = np.zeros((*np.array(img).shape[:2], 4), dtype=np.float32)
    for mask in results['masks']:
        mask_bool = mask.cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.array(mask).astype(bool)
        color = np.concatenate([np.random.random(3), [0.5]])
        overlay[mask_bool] = color
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f'{len(results["masks"])} instances')
    axes[1, col].axis('off')

plt.suptitle(f'Text prompt "{TEXT_PROMPT}" across zoom levels', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Point-prompted segmentation (SAM3 Tracker)

Same point/box interface as SAM2 — click on specific things to segment.
- Positive points (label=1): "segment this"
- Negative points (label=0): "not this"

Coordinates are in pixels (0,0 = top-left). The image is 640x640.

In [ ]:
def segment_with_points(
    image: Image.Image,
    points: list[tuple[int, int]],
    labels: list[int],
    title: str = 'Point prompt',
):
    """Segment using point prompts via SAM3 Tracker."""
    # Format: [image_dim, object_dim, point_per_object_dim, coordinates]
    input_points = [[points]]
    input_labels = [[labels]]

    inputs = sam3_tracker_processor(
        images=image,
        input_points=input_points,
        input_labels=input_labels,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        outputs = sam3_tracker(**inputs)

    masks = sam3_tracker_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs['original_sizes']
    )[0]

    # masks shape: [num_objects, num_masks, H, W]
    num_masks = masks.shape[1]
    fig, axes = plt.subplots(1, num_masks + 1, figsize=(5 * (num_masks + 1), 5))

    # Original with points
    axes[0].imshow(image)
    for (x, y), label in zip(points, labels):
        color = 'lime' if label == 1 else 'red'
        axes[0].plot(x, y, marker='*', markersize=15, color=color,
                     markeredgecolor='white', markeredgewidth=1)
    axes[0].set_title('Input points')
    axes[0].axis('off')

    # Mask predictions
    for i in range(num_masks):
        mask_bool = masks[0, i].numpy().astype(bool)
        axes[i + 1].imshow(image)
        overlay = np.zeros((*mask_bool.shape, 4), dtype=np.float32)
        overlay[mask_bool] = [0.2, 0.6, 1.0, 0.5]
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f'Mask {i+1}')
        axes[i + 1].axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

    return masks


img = images[19]

# ── CHANGE THESE POINTS ──
segment_with_points(
    img,
    points=[(320, 320)],
    labels=[1],
    title='Rooftop segmentation (center point)',
)

## 6. Multi-point prompts — refine the mask

In [ ]:
# Positive + negative points to isolate the house from yard
segment_with_points(
    images[19],
    points=[
        (320, 300),   # on the roof (positive)
        (320, 280),   # also on the roof (positive)
        (320, 430),   # backyard / grass (negative)
        (200, 320),   # neighbor's lot (negative)
    ],
    labels=[1, 1, 0, 0],
    title='House isolation: roof (+) vs yard/neighbors (-)',
)

## 7. Box-prompted segmentation

In [ ]:
def segment_with_box(
    image: Image.Image,
    box: tuple[int, int, int, int],
    title: str = 'Box prompt',
):
    """Segment using a bounding box via SAM3 Tracker."""
    input_boxes = [[list(box)]]

    inputs = sam3_tracker_processor(
        images=image,
        input_boxes=input_boxes,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        outputs = sam3_tracker(**inputs)

    masks = sam3_tracker_processor.post_process_masks(
        outputs.pred_masks.cpu(), inputs['original_sizes']
    )[0]

    num_masks = masks.shape[1]
    fig, axes = plt.subplots(1, num_masks + 1, figsize=(5 * (num_masks + 1), 5))

    # Original with box
    axes[0].imshow(image)
    rect = mpatches.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1],
        linewidth=2, edgecolor='lime', facecolor='none',
    )
    axes[0].add_patch(rect)
    axes[0].set_title('Input box')
    axes[0].axis('off')

    for i in range(num_masks):
        mask_bool = masks[0, i].numpy().astype(bool)
        axes[i + 1].imshow(image)
        overlay = np.zeros((*mask_bool.shape, 4), dtype=np.float32)
        overlay[mask_bool] = [1.0, 0.4, 0.1, 0.5]
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f'Mask {i+1}')
        axes[i + 1].axis('off')

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

    return masks


# ── CHANGE THE BOX ──
segment_with_box(
    images[19],
    box=(220, 220, 420, 420),
    title='Property bounding box',
)

## 8. Text + negative box — exclude specific regions

SAM3 lets you combine text prompts with negative boxes to exclude regions from segmentation.

In [ ]:
def segment_text_with_negative_box(
    image: Image.Image,
    text: str,
    exclude_box: tuple[int, int, int, int],
    title: str = '',
):
    """Segment with text but exclude a bounding box region."""
    input_boxes = [[list(exclude_box)]]
    input_boxes_labels = [[0]]  # 0 = negative

    inputs = sam3_processor(
        images=image,
        text=text,
        input_boxes=input_boxes,
        input_boxes_labels=input_boxes_labels,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    masks = results['masks']
    scores = results['scores']
    n_masks = len(masks)
    n_cols = min(n_masks + 1, 5)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]

    axes[0].imshow(image)
    rect = mpatches.Rectangle(
        (exclude_box[0], exclude_box[1]),
        exclude_box[2] - exclude_box[0], exclude_box[3] - exclude_box[1],
        linewidth=2, edgecolor='red', facecolor='none', linestyle='--',
    )
    axes[0].add_patch(rect)
    axes[0].set_title(f'Text: "{text}" (red = excluded)')
    axes[0].axis('off')

    for i in range(min(n_masks, n_cols - 1)):
        mask = masks[i]
        mask_bool = mask.cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.array(mask).astype(bool)
        axes[i + 1].imshow(image)
        overlay = np.zeros((*mask_bool.shape[:2], 4), dtype=np.float32)
        overlay[mask_bool] = [0.2, 0.8, 0.2, 0.5]
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f'Instance {i+1} (score: {float(scores[i]):.3f})')
        axes[i + 1].axis('off')

    display_title = title or f'"{text}" excluding box region'
    plt.suptitle(f'{display_title} — {n_masks} instance(s)', fontsize=14)
    plt.tight_layout()
    plt.show()

    return results


# Example: find all rooftops except the center property
segment_text_with_negative_box(
    images[19],
    text='rooftop',
    exclude_box=(250, 250, 400, 400),
    title='Neighbor rooftops (excluding center property)',
)

## 9. Automatic mask generation (pipeline)

SAM3 also supports automatic mask generation via the HuggingFace pipeline.

In [ ]:
from transformers import pipeline


def show_auto_masks(image: Image.Image, masks: list, title: str = '', ax=None):
    """Overlay automatic segmentation masks on image."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    if not masks:
        ax.set_title(f'{title} (no masks found)')
        ax.axis('off')
        return

    sorted_masks = sorted(masks, key=lambda x: x['mask'].sum(), reverse=True)
    overlay = np.zeros((*np.array(image).shape[:2], 4), dtype=np.float32)

    for mask_data in sorted_masks:
        seg = np.array(mask_data['mask'])
        mask_bool = seg.astype(bool)
        color = np.concatenate([np.random.random(3), [0.5]])
        overlay[mask_bool] = color

    ax.imshow(overlay)
    ax.set_title(f'{title} ({len(masks)} masks)')
    ax.axis('off')


mask_generator = pipeline('mask-generation', model='facebook/sam3', device=device)

ZOOM = 19
img = images[ZOOM]

print(f'Generating masks for zoom {ZOOM}...')
auto_masks = mask_generator(img, points_per_batch=64)
print(f'Found {len(auto_masks["masks"])} masks')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.imshow(img)
ax1.set_title(f'Original (zoom {ZOOM})')
ax1.axis('off')
show_auto_masks(img, [{'mask': m} for m in auto_masks['masks']], title=f'Auto masks (zoom {ZOOM})', ax=ax2)
plt.tight_layout()
plt.show()

## 10. Text prompt sweep — compare different concepts

Side-by-side comparison of different text prompts on the same image.

In [ ]:
# ── CHANGE THESE PROMPTS ──
TEXT_PROMPTS = ['rooftop', 'driveway', 'tree', 'road', 'car', 'grass']

img = images[19]
n_prompts = len(TEXT_PROMPTS)
cols = min(n_prompts, 3)
rows = (n_prompts + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
axes = np.array(axes).flatten()

for i, text in enumerate(TEXT_PROMPTS):
    inputs = sam3_processor(images=img, text=text, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    axes[i].imshow(img)
    overlay = np.zeros((*np.array(img).shape[:2], 4), dtype=np.float32)
    for mask in results['masks']:
        mask_bool = mask.cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.array(mask).astype(bool)
        color = np.concatenate([np.random.random(3), [0.5]])
        overlay[mask_bool] = color
    axes[i].imshow(overlay)
    axes[i].set_title(f'"{text}" — {len(results["masks"])} instances')
    axes[i].axis('off')

# Hide unused axes
for j in range(n_prompts, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Text prompt comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Mask statistics

In [ ]:
img = images[19]
img_w, img_h = img.size

print(f'Image size: {img_w}x{img_h}')
print(f'{"":-<60}')

for text in ['rooftop', 'driveway', 'tree', 'car', 'road']:
    inputs = sam3_processor(images=img, text=text, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    masks = results['masks']
    scores = results['scores']
    print(f'\n"{text}" — {len(masks)} instance(s):')
    for i, (mask, score) in enumerate(zip(masks, scores)):
        mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else np.array(mask)
        area = mask_np.astype(bool).sum()
        area_pct = area / (img_w * img_h) * 100
        print(f'  Instance {i+1}: area={area:6d}px ({area_pct:5.1f}%) score={float(score):.3f}')

## 12. Try a different address

In [ ]:
# ── CHANGE ADDRESS ──
NEW_LAT, NEW_LON = 43.6532, -79.3832  # downtown Toronto
NEW_ZOOM = 19

new_img = fetch_satellite(NEW_LAT, NEW_LON, zoom=NEW_ZOOM)

# Text-prompted segmentation on the new location
prompts = ['building', 'road', 'car']
fig, axes = plt.subplots(1, len(prompts) + 1, figsize=(5 * (len(prompts) + 1), 5))

axes[0].imshow(new_img)
axes[0].set_title(f'Satellite ({NEW_LAT:.4f}, {NEW_LON:.4f})')
axes[0].axis('off')

for i, text in enumerate(prompts):
    inputs = sam3_processor(images=new_img, text=text, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = sam3_model(**inputs)
    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=inputs.get('original_sizes').tolist(),
    )[0]

    axes[i + 1].imshow(new_img)
    overlay = np.zeros((*np.array(new_img).shape[:2], 4), dtype=np.float32)
    for mask in results['masks']:
        mask_bool = mask.cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.array(mask).astype(bool)
        color = np.concatenate([np.random.random(3), [0.5]])
        overlay[mask_bool] = color
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f'"{text}" — {len(results["masks"])} instances')
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()